# Cahn-Hilliard equation on the sphere

A general nonlinear PDE for a function $u\in U$ can be defined by operator / map $R:U\to V^*$ so that you can test

$$
\langle R(u),v \rangle = 0
$$

for all $v\in V$.


In [ ]:
%%capture
try:
    import dolfin
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/fenics-install-release-real.sh" -O "/tmp/fenics-install.sh" && bash "/tmp/fenics-install.sh"
    import dolfin

In [ ]:
!git clone https://github.com/dpeschka/cahn-hilliard-sphere.git
folder = 'cahn-hilliard-sphere/'

In [ ]:
from dolfin import *
import logging
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# Load original mesh
temp = Mesh(folder + 'data/sphere.xml')

# Extract boundary and refine
mesh = BoundaryMesh(temp, 'exterior')
mesh = refine(mesh)

# Project vertices back to the sphere (radius 1)
# This is necessary because linear refinement places new points on segments
x = mesh.coordinates()
x[:] /= np.sqrt(np.sum(x**2, axis=1))[:, np.newaxis]

# Set logging levels to reduce noise
logging.getLogger("FFC").setLevel(logging.ERROR)
logging.getLogger("UFL_LEGACY").setLevel(logging.ERROR)
logging.getLogger("UFL").setLevel(logging.ERROR)
set_log_level(LogLevel.ERROR)

In [ ]:
# FE definitions
L = 1.0
Λ = 1.0
ε = 0.1
T = 1.0

# mesh = IntervalMesh(128,0,L)
FE   = FiniteElement("P", mesh.ufl_cell(), 1)   # scalar element
Q    = FunctionSpace(mesh,MixedElement([FE,FE]))# mixed space


def W(φ):
  return 18*Λ*(φ*(1-φ))**2

# energy
def energy(q):
    φ,μ = split(q)
    E  = ε/2 * inner(grad(φ),grad(φ))*dx
    E += 1/ε * W(φ)*dx
    return E


# single time step
def evolve(old_q, τ):
    # set up function spaces
    q = Function(Q)
    v = TestFunction(Q)
    φ,μ = split(q)
    vφ,vμ = split(v)
    old_φ,_ = split(old_q)

    # define energy
    E = energy(q)
    m = 1.0

    # define weak form
    Res  = μ*vφ*dx - derivative(E, q, v)
    Res += ε**2*τ*m*inner(grad(μ),grad(vμ))*dx + vμ*(φ-old_φ)*dx

    bc = []

    Jac     = derivative(Res, q)
    problem = NonlinearVariationalProblem(Res, q, bc, Jac)
    solver  = NonlinearVariationalSolver(problem)

    prm = solver.parameters
    prm['newton_solver']['error_on_nonconvergence'] = False
    prm['newton_solver']['report'] = False
    prm['newton_solver']['absolute_tolerance'] = 1e-5
    prm['newton_solver']['relative_tolerance'] = 1e-5

    q.assign(old_q)
    iterations, converged = solver.solve()

    return q,iterations,converged

# initial data
idata = Expression(("((float)(rand()))","0"),degree=2,ε=ε)
old_q = interpolate(idata,Q)
old_q,it,conv = evolve(old_q,1e-4)
print('init:',it,conv)
times = []
energies = []
sols = []


# initial time stepping, later adaptive
t = 0
τ = 0.3e-2
n_steps = 10
dt = Constant(τ)

f = File('data/solution.pvd')

for i in tqdm(range(n_steps)):
  dt.assign(τ)

  q,it,conv = evolve(old_q, dt)



  if conv:
    phi,mu = q.split()
    phi.rename("phi","phi")
    f << (phi,t)

    t += τ
    old_q.assign(q)
    if (i % 2 == 0):
      times.append(t)
      sols.append(q)
    if it < 3:
      τ *= 1.1
    if it > 4:
      τ *= 0.95
  else:
    τ *= 0.75
  if t>T:
    break

print(t)


In [ ]:
!zip data.zip data/*

In [ ]:
print(mesh.num_vertices())

In [ ]:
from dolfin import *
mesh = Mesh('data/sphere.xml')

mesh1 = BoundaryMesh(mesh, 'exterior')
V1 = FunctionSpace(mesh1, 'P', 1)

u1 = interpolate(Expression('x[0]', degree=1), V1)

f1 = File('data/sphere_boundary.pvd')
f1 << u1

print(mesh.num_vertices())
print(mesh1.num_vertices())